##### 1. Experiment with hyperparameter tuning for a GPT model and document the effects on model performance.
● Choose a set of hyperparameters to tune (e.g., learning rate, batch size, numberof epochs).

● Conduct experiments to observe the impact of different hyperparameter values on model performance.

● Summarize the results in a detailed report (1000-1200 words), including plots or tables to visualize performance changes.
##### 2. Compare and contrast different variants of GPT models in a written report, including their applications and capabilities.
● Research different variants of GPT models (e.g., GPT-2, GPT-3, GPT-Neo, GPT-J).

● Write a comprehensive report (1500-2000 words) comparing their architectures, training data, and performance.

● Discuss the specific applications and capabilities of each variant, providing realworld examples where possible.

In [3]:
import torch 
from transformers import (GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling,)
from datasets import load_dataset 
import math 
import matplotlib.pyplot as plt 
import pandas as pd

In [4]:
# checking gpu availabilty 
device = "cuda" if torch.cuda.is_available() else "cpu" 
print (f"Using {device} device")

Using cuda device


##### Loading the dataset and preparing the dataset

In [5]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
print("Dataset structure:")
print(dataset)
print(f"\nTraining examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['validation'])}")
print(f"\nSample text:")
print(dataset["train"][100]["text"][:300])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Dataset structure:
DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

Training examples: 36718
Validation examples: 3760

Sample text:
 96 ammunition packing boxes 



##### tokenizing the dataset

In [6]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
# gpt 2 doesnt have padding toekn by default so i set one 
tokenizer.pad_token = tokenizer.eos_token 
BLOCK_SIZE = 128

def tokenize_function(examples):
    tokenized = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=BLOCK_SIZE)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_dataset = tokenized_dataset.filter(lambda x: any(t != tokenizer.pad_token_id for t in x["input_ids"]))

print(f"Tokenized training examples: {len(tokenized_dataset['train'])}")
print(f"Tokenized validation examples: {len(tokenized_dataset['validation'])}")
    

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

Tokenized training examples: 23767
Tokenized validation examples: 2461


##### creating a training function

In [7]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False,)


##### Training function (i will be running five experiments so i set a training function so i wont need to repeat the function anytime im trying other parameters)

In [8]:
def run_experiment(experiment_name, learning_rate, batch_size, num_epochs):

    print(f"\n{'='*60}")
    print(f"Running: {experiment_name}")
    print(f"  Learning Rate: {learning_rate}")
    print(f"  Batch Size:    {batch_size}")
    print(f"  Epochs:        {num_epochs}")
    print(f"{'='*60}\n")
    
    model = GPT2LMHeadModel.from_pretrained("gpt2")
    model.config.pad_token_id = tokenizer.pad_token_id
    
    training_args = TrainingArguments(
        output_dir=f"./results/{experiment_name}",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        weight_decay=0.01,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        data_collator=data_collator,
    )
    
    trainer.train()
    
    eval_results = trainer.evaluate()
    eval_loss = eval_results["eval_loss"]
    perplexity = math.exp(eval_loss)
    
    print(f"\n--- Results for {experiment_name} ---")
    print(f"  Eval Loss:   {eval_loss:.4f}")
    print(f"  Perplexity:  {perplexity:.2f}")
    
    return {
        "experiment": experiment_name,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "epochs": num_epochs,
        "eval_loss": round(eval_loss, 4),
        "perplexity": round(perplexity, 2),
    }

all_results = []

##### Baseline- the starting settings



In [7]:
baseline = run_experiment("Baseline", learning_rate=5e-5, batch_size=8, num_epochs=3)
all_results.append(baseline)



Running: Baseline
  Learning Rate: 5e-05
  Batch Size:    8
  Epochs:        3



config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.338500,3.346378
2,3.088114,3.344941
3,2.988951,3.352178



--- Results for Baseline ---
  Eval Loss:   3.3522
  Perplexity:  28.56


##### Experiment 1- Low learning rate

In [8]:
exp_a1 = run_experiment("Exp_A1_Low_LR", learning_rate=1e-5, batch_size=8, num_epochs=3)
all_results.append(exp_a1)


Running: Exp_A1_Low_LR
  Learning Rate: 1e-05
  Batch Size:    8
  Epochs:        3



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
1,3.453148,3.380861
2,3.332605,3.359735
3,3.320699,3.354295



--- Results for Exp_A1_Low_LR ---
  Eval Loss:   3.3543
  Perplexity:  28.63


##### Experiment 2 - high learning rate

In [9]:
exp_a2 = run_experiment("Exp_A2_High_LR", learning_rate=5e-4, batch_size=8, num_epochs=3)
all_results.append(exp_a2)


Running: Exp_A2_High_LR
  Learning Rate: 0.0005
  Batch Size:    8
  Epochs:        3



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
1,3.592124,3.797683
2,2.862806,3.790728
3,2.213489,3.979901



--- Results for Exp_A2_High_LR ---
  Eval Loss:   3.9799
  Perplexity:  53.51


##### Experiment 3- Small batch size

In [10]:
exp_b1 = run_experiment("Exp_B1_Low_LR", learning_rate=5e-5, batch_size=4, num_epochs=3)
all_results.append(exp_b1)


Running: Exp_B1_Low_LR
  Learning Rate: 5e-05
  Batch Size:    4
  Epochs:        3



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
1,3.274964,3.355665
2,2.903529,3.351845
3,2.835566,3.369652



--- Results for Exp_B1_Low_LR ---
  Eval Loss:   3.3697
  Perplexity:  29.07


##### Experiment 4 - Large batch size

In [9]:
exp_b2 = run_experiment("Exp_B2_High_LR", learning_rate=5e-5, batch_size=16, num_epochs=3)
all_results.append(exp_b2)


Running: Exp_B2_High_LR
  Learning Rate: 5e-05
  Batch Size:    16
  Epochs:        3



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.346700,3.338631
2,3.152672,3.329874
3,3.064710,3.335355



--- Results for Exp_B2_High_LR ---
  Eval Loss:   3.3354
  Perplexity:  28.09


##### Experiment 5 - few epochs

In [10]:
exp_c1 = run_experiment("Exp_C1_Few_Epochs", learning_rate=5e-5, batch_size=8, num_epochs=1)
all_results.append(exp_c1)


Running: Exp_C1_Few_Epochs
  Learning Rate: 5e-05
  Batch Size:    8
  Epochs:        1



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
1,3.362070,3.343141



--- Results for Exp_C1_Few_Epochs ---
  Eval Loss:   3.3431
  Perplexity:  28.31


##### Experiment 6 - many epochs

In [11]:
exp_c2 = run_experiment("Exp_C2_Many_Epochs", learning_rate=5e-5, batch_size=8, num_epochs=5)
all_results.append(exp_c2)


Running: Exp_C2_Many_Epochs
  Learning Rate: 5e-05
  Batch Size:    8
  Epochs:        5



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
1,3.338444,3.350507
2,3.081570,3.353390
3,2.952520,3.370003
4,2.816496,3.388435
5,2.718729,3.405571



--- Results for Exp_C2_Many_Epochs ---
  Eval Loss:   3.4056
  Perplexity:  30.13


##### Improved data preparation to try and improve result

In [13]:
from transformers import get_linear_schedule_with_warmup

##### Concatenating all text, tokenizing it, then splitting into BLOCK_SIZE chunks.No padding waste this time

In [14]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

BLOCK_SIZE = 512  
def tokenize_and_chunk(examples):
    concatenated = tokenizer(examples["text"], truncation=False, padding=False)
    all_ids = []
    for ids in concatenated["input_ids"]:
        all_ids.extend(ids)
    
    total_length = (len(all_ids) // BLOCK_SIZE) * BLOCK_SIZE  # drop remainder
    
    result = {
        "input_ids": [all_ids[i:i + BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)],
        "attention_mask": [[1] * BLOCK_SIZE for _ in range(0, total_length, BLOCK_SIZE)],
    }
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = dataset.map(
    tokenize_and_chunk,
    batched=True,
    remove_columns=["text"],
    batch_size=1000,
)

print(f"Training chunks: {len(tokenized_dataset['train'])}")
print(f"Validation chunks: {len(tokenized_dataset['validation'])}")
print(f"Each chunk: {BLOCK_SIZE} tokens (no padding!)")

Map:   0%|          | 0/4358 [00:00<?, ? examples/s]

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

Training chunks: 4656
Validation chunks: 481
Each chunk: 512 tokens (no padding!)


##### Improved training function

##### Improved training with:
#####    - Learning rate warmup (gradually increases LR at the start)
#####    - Cosine decay schedule (slowly decreases LR toward the end)
#####    - Gradient accumulation (simulates larger batches)

In [15]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

def run_improved_experiment(experiment_name, learning_rate, batch_size, num_epochs, warmup_steps=100):

    print(f"\n{'='*60}")
    print(f"Running: {experiment_name}")
    print(f"  Learning Rate: {learning_rate}")
    print(f"  Batch Size:    {batch_size}")
    print(f"  Epochs:        {num_epochs}")
    print(f"  Warmup Steps:  {warmup_steps}")
    print(f"{'='*60}\n")
    
    model = GPT2LMHeadModel.from_pretrained("gpt2")
    model.config.pad_token_id = tokenizer.pad_token_id
    
    training_args = TrainingArguments(
        output_dir=f"./results_v2/{experiment_name}",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        weight_decay=0.01,
        warmup_steps=warmup_steps,             
        lr_scheduler_type="cosine",            
        gradient_accumulation_steps=4,         
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        data_collator=data_collator,
    )
    
    trainer.train()
    
    eval_results = trainer.evaluate()
    eval_loss = eval_results["eval_loss"]
    perplexity = math.exp(eval_loss)
    
    print(f"\n--- Results for {experiment_name} ---")
    print(f"  Eval Loss:   {eval_loss:.4f}")
    print(f"  Perplexity:  {perplexity:.2f}")
    
    return {
        "experiment": experiment_name,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "epochs": num_epochs,
        "eval_loss": round(eval_loss, 4),
        "perplexity": round(perplexity, 2),
    }

improved_results = []

In [16]:
improved_baseline = run_improved_experiment(
    "Improved_Baseline", learning_rate=5e-5, batch_size=8, num_epochs=3
)
improved_results.append(improved_baseline)


Running: Improved_Baseline
  Learning Rate: 5e-05
  Batch Size:    8
  Epochs:        3
  Warmup Steps:  100



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
1,3.394535,3.168171
2,3.200513,3.136298
3,3.128296,3.131426



--- Results for Improved_Baseline ---
  Eval Loss:   3.1314
  Perplexity:  22.91


##### Visualization for results

In [2]:
import pandas as pd

In [3]:
#creating a dataframe from my results
results_data = {
    "Experiment": [
        "Baseline", "Low LR\n(1e-5)", "High LR\n(5e-4)", 
        "Small Batch\n(4)", "Large Batch\n(16)", 
        "Few Epochs\n(1)", "Many Epochs\n(5)", "Improved\nBaseline"
    ],
    "Eval Loss": [3.3522, 3.3543, 3.9799, 3.3697, 3.3354, 3.3431, 3.4056, 3.1314],
    "Perplexity": [28.56, 28.63, 53.51, 29.07, 28.09, 28.31, 30.13, 22.91],
}

df = pd.DataFrame(results_data)
print(df.to_string(index=False))

        Experiment  Eval Loss  Perplexity
          Baseline     3.3522       28.56
    Low LR\n(1e-5)     3.3543       28.63
   High LR\n(5e-4)     3.9799       53.51
  Small Batch\n(4)     3.3697       29.07
 Large Batch\n(16)     3.3354       28.09
   Few Epochs\n(1)     3.3431       28.31
  Many Epochs\n(5)     3.4056       30.13
Improved\nBaseline     3.1314       22.91


##### Perplexity comparson bar chart